# Step 5: Operational Recommendations

This notebook consolidates the main findings from the prior notebooks into an executive-ready summary.

Run notebooks 2 through 4 first so the analysis files exist in `outputs/analysis/`.


In [1]:
import pandas as pd


In [2]:
from pathlib import Path


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, current.parent):
        if (candidate / "orders.csv").exists() and (candidate / "products.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root that contains the Instacart CSV files.")


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
ANALYSIS_DIR = OUTPUT_DIR / "analysis"
FIGURE_DIR = OUTPUT_DIR / "figures"
QUALITY_DIR = OUTPUT_DIR / "quality"

for path in (OUTPUT_DIR, ANALYSIS_DIR, FIGURE_DIR, QUALITY_DIR):
    path.mkdir(parents=True, exist_ok=True)


In [3]:
demand_summary = pd.read_csv(ANALYSIS_DIR / "demand_summary.csv")
top_products = pd.read_csv(ANALYSIS_DIR / "top_products.csv")
association_rules = pd.read_csv(ANALYSIS_DIR / "association_rules.csv")
staffing_plan = pd.read_csv(ANALYSIS_DIR / "staffing_plan.csv")

demand_summary


,metric,value,order_count
0,busiest_hour,10,282470
1,slowest_hour,3,5343
2,busiest_day,Tuesday,542103
3,slowest_day,Sunday,442564


In [4]:
best_rule = association_rules.sort_values(["lift", "confidence"], ascending=False).iloc[0]
top_product = top_products.sort_values("order_count", ascending=False).iloc[0]
peak_staff = staffing_plan.sort_values("recommended_shoppers", ascending=False).iloc[0]

executive_summary = pd.DataFrame(
    [
        {
            "focus_area": "Demand timing",
            "finding": f"The busiest operating window is {demand_summary.loc[demand_summary['metric'] == 'busiest_day', 'value'].iloc[0]} and the peak hour is {demand_summary.loc[demand_summary['metric'] == 'busiest_hour', 'value'].iloc[0]}.",
            "recommended_action": "Prioritize staffing, picking capacity, and slot availability during the strongest demand windows.",
        },
        {
            "focus_area": "Top products",
            "finding": f"{top_product['product_name']} appears as the most frequently purchased product in the dataset slice used for basket analysis.",
            "recommended_action": "Protect availability for top-volume products and use them as anchor items in inventory planning.",
        },
        {
            "focus_area": "Cross-sell",
            "finding": f"A strong rule is '{best_rule['antecedents']}' -> '{best_rule['consequents']}' with lift {best_rule['lift']:.2f}.",
            "recommended_action": "Use the top rules for recommendation widgets, bundle prompts, and merchandising tests.",
        },
        {
            "focus_area": "Staffing",
            "finding": f"The highest suggested staffing level in the current model is {int(peak_staff['recommended_shoppers'])} shoppers at hour {int(peak_staff['order_hour'])} on {peak_staff['day_name']}.",
            "recommended_action": "Use the staffing table as the operational baseline, then recalibrate costs and shopper productivity assumptions with business input.",
        },
    ]
)

executive_summary


,focus_area,finding,recommended_action
0,Demand timing,The busiest operating window is Tuesday and th...,"Prioritize staffing, picking capacity, and slo..."
1,Top products,Banana appears as the most frequently purchase...,Protect availability for top-volume products a...
2,Cross-sell,A strong rule is 'Organic Garlic' -> 'Organic ...,"Use the top rules for recommendation widgets, ..."
3,Staffing,The highest suggested staffing level in the cu...,Use the staffing table as the operational base...


In [5]:
executive_summary.to_csv(ANALYSIS_DIR / "executive_summary.csv", index=False)
executive_summary


,focus_area,finding,recommended_action
0,Demand timing,The busiest operating window is Tuesday and th...,"Prioritize staffing, picking capacity, and slo..."
1,Top products,Banana appears as the most frequently purchase...,Protect availability for top-volume products a...
2,Cross-sell,A strong rule is 'Organic Garlic' -> 'Organic ...,"Use the top rules for recommendation widgets, ..."
3,Staffing,The highest suggested staffing level in the cu...,Use the staffing table as the operational base...


## Step 5 Outcome

After running this notebook you will have:
- a concise summary of demand, basket, and staffing insights
- a CSV export at `outputs/analysis/executive_summary.csv`
- a final notebook suitable for presentation prep and report writing
